In [85]:
import pandas as pd

In [86]:
df = pd.read_csv("Ames Housing/AmesHousing.csv")

In [87]:
df.dtypes.value_counts()

object     43
int64      28
float64    11
Name: count, dtype: int64

In [88]:
print("total rows count: ", len(df))
num_missing = df.select_dtypes(exclude=["object"]).isnull().sum()
num_missing[num_missing > 0].sort_values(ascending=False)


total rows count:  2930


Lot Frontage      490
Garage Yr Blt     159
Mas Vnr Area       23
Bsmt Half Bath      2
Bsmt Full Bath      2
BsmtFin SF 1        1
BsmtFin SF 2        1
Total Bsmt SF       1
Bsmt Unf SF         1
Garage Cars         1
Garage Area         1
dtype: int64

In [89]:
# --- 1. Numerical "Pseudo-Missing" (Fill with 0) ---
# These columns have NaN when the feature (Garage, Basement, etc.) doesn't exist
# Updated with spaces to match your dataset version
num_zero_cols = [
    'Garage Area', 'Garage Cars', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 
    'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Mas Vnr Area'
]
df[num_zero_cols] = df[num_zero_cols].fillna(0)

In [90]:
print("total rows count: ", len(df))
cat_missing = df.select_dtypes(include=["object"]).isnull().sum()
cat_missing[cat_missing > 0].sort_values(ascending=False)


total rows count:  2930


Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Garage Qual        159
Garage Finish      159
Garage Cond        159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Qual           80
BsmtFin Type 1      80
Bsmt Cond           80
Electrical           1
dtype: int64

In [91]:
# --- 2. Categorical "Pseudo-Missing" (Fill with "None") ---
# These columns have NaN when the house doesn't have an Alley, Pool, Fence, etc.
cat_none_cols = [
    'Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Mas Vnr Type', 
    'Fireplace Qu', 'Garage Qual', 'Garage Finish', 'Garage Cond', 
    'Garage Type', 'Bsmt Exposure', 'BsmtFin Type 2', 'Bsmt Qual', 
    'BsmtFin Type 1', 'Bsmt Cond'
]

df[cat_none_cols] = df[cat_none_cols].fillna('None')

In [92]:
# verify missing value again
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

Lot Frontage     490
Garage Yr Blt    159
Electrical         1
dtype: int64

In [93]:
df["Lot Frontage"].value_counts().sort_values() # the length of the property (plot) that faces the street
# so fill by near area length (means Neighborhood) and as mean is affect by outliners i used median

Lot Frontage
155.0      1
126.0      1
200.0      1
168.0      1
111.0      1
        ... 
75.0     105
50.0     117
70.0     133
80.0     137
60.0     276
Name: count, Length: 128, dtype: int64

In [94]:
df["Lot Frontage"] = df.groupby("Neighborhood")["Lot Frontage"].transform(lambda x: x.fillna(x.median()))

C:\Users\kkp18\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\kkp18\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [95]:
df["Garage Yr Blt"].value_counts()

Garage Yr Blt
2005.0    142
2006.0    115
2007.0    115
2004.0     99
2003.0     92
         ... 
1908.0      1
1933.0      1
2207.0      1
1943.0      1
1919.0      1
Name: count, Length: 103, dtype: int64

In [96]:
# missing fill with the house year build
df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(df["Year Built"])

In [97]:
print(df["Electrical"].value_counts()) # Type of electrical system - categorical
# so missing fill with mode
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])
print("after mode: ", df["Electrical"].value_counts())

Electrical
SBrkr    2682
FuseA     188
FuseF      50
FuseP       8
Mix         1
Name: count, dtype: int64
after mode:  Electrical
SBrkr    2683
FuseA     188
FuseF      50
FuseP       8
Mix         1
Name: count, dtype: int64


In [98]:
# verify missing value again
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

Lot Frontage    3
dtype: int64

In [99]:
# 2. Fallback: Fill any remaining NaNs with the global median of the whole dataset
# (This handles neighborhoods that had NO data at all)
df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

In [100]:
# verify missing value again
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [101]:
# Removing outliers (based on EDA)
df = df.drop(df[(df["Gr Liv Area"] > 4000) & (df["SalePrice"] < 300000)].index)

In [102]:
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,None,IR1,Lvl,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,None,Reg,Lvl,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,None,IR1,Lvl,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,None,Reg,Lvl,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,None,IR1,Lvl,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


In [103]:
# MS SubClass is a category not a numbers
df["MS SubClass"] = df["MS SubClass"].apply(str)
# If we leave the codes as numbers, the model tries to do math on them (like 120 is "better" than 60).
# By converting to strings, we tell the model: "This is a name, not a measurement."

In [104]:

# 2. Month Sold: Month 12 isn't "12 times bigger" than Month 1.
# It's better to treat them as categories for seasonality.
df['Mo Sold'] = df['Mo Sold'].apply(str)

In [105]:
# 3. Year Sold: Treating the year as a category helps the model
# pick up on specific market conditions for that year.
df['Yr Sold'] = df['Yr Sold'].apply(str)

In [106]:
# Verify the types have changed to 'object' (string)
print(df[['MS SubClass', 'Mo Sold', 'Yr Sold']].dtypes)

MS SubClass    object
Mo Sold        object
Yr Sold        object
dtype: object


In [107]:
# --- Check for Duplicate Rows ---
df.duplicated().sum()

np.int64(0)

In [108]:
# 2. Create Total Square Footage
# This is often the #1 predictor of price in a model
df['Total SF'] = df['Total Bsmt SF'] + df['1st Flr SF'] + df['2nd Flr SF']

In [109]:
df.to_csv("Ames Housing/AmesHousing-Cleaned.csv", index=False)